In [3]:
from ase.io import read,write
from ase.visualize import view
from ase.build import surface
cif_dir = '../cif/output_file'
metals = ['TiN','VN','ZrN','ScN','NbN']

for metal in metals:
    cif = read(f'{cif_dir}/{metal}_relaxed.extxyz')

    slab = surface(cif, (1, 0, 0), layers=2, vacuum=10.0)
    view(slab)
    #write(f'./slim/{metal}_slim_slab.xyz', slab)

In [13]:
view(slab)

<Popen: returncode: None args: ['/home/ameer_ubuntu/miniforge3/envs/qe/bin/p...>

In [17]:
# FULL SLAB MAKER
from ase.io import read,write
from ase.visualize import view
from ase.build import surface
from pathlib import Path

cif_dir = '../cif/output_file'
metals = ['TiN','VN','ZrN','ScN','NbN']

out_dir = Path('./input_slab_144')
out_dir.mkdir(exist_ok=True)

for metal in metals:
    cif = read(f'{cif_dir}/{metal}_relaxed.extxyz')

    slab = surface(cif, (1, 0, 0), layers=2, vacuum=10.0)
    slab = slab.repeat((3, 3, 1))
    view(slab)
    #write(out_dir / f'{metal}_144_slab.xyz', slab)


In [15]:
from ase.io import read, write
from pathlib import Path
from collections import Counter

# Process ALL .xyz files in any directory matching `input_slab*`
base_dirs = sorted(Path('.').glob('input_slab*'))
xyz_files = [p for d in base_dirs for p in sorted(d.glob('*.xyz'))]

if not xyz_files:
    print('No .xyz files found in input_slab* folders.')
else:
    # Safety: set commit=True to actually overwrite files
    commit = True  # <- change to True to write changes ****

    processed = 0
    changed = 0

    for file_path in xyz_files:
        processed += 1
        try:
            atoms = read(file_path)
        except Exception as exc:
            print(f"Skipping {file_path.name}: read error: {exc}")
            continue

        symbols = atoms.get_chemical_symbols()
        counts = Counter(symbols)
        species_set = set(counts.keys())

        # If it's a binary nitride (Metal + N) prefer Metal first
        if 'N' in species_set and len(species_set) == 2:
            other = next(s for s in species_set if s != 'N')
            species_order = [other, 'N']
        else:
            # otherwise order by frequency (most common first), tie by symbol
            species_order = sorted(counts.keys(), key=lambda s: (-counts[s], s))

        # build new ordering of indices
        new_indices = []
        for s in species_order:
            new_indices += [i for i, sym in enumerate(symbols) if sym == s]

        # append any remaining atoms (shouldn't normally be any)
        remaining = [i for i, sym in enumerate(symbols) if sym not in species_order]
        new_indices += remaining

        new_atoms = atoms[new_indices]

        # detect whether ordering actually changed (compare symbol lists)
        if [a.symbol for a in atoms] == [a.symbol for a in new_atoms]:
            print(f"{file_path.name}: ordering already correct ({species_order})")
            continue

        if commit:
            try:
                write(file_path, new_atoms)
                print(f"{file_path.name}: rewritten with order {species_order}")
            except Exception as exc:
                print(f"Failed to write {file_path.name}: {exc}")
        else:
            print(f"{file_path.name}: DRY-RUN — would reorder -> {species_order}")

        changed += 1

    print(f"\nSummary: processed={processed}, would_change_or_changed={changed}, commit={commit}")

NbN_108_slab.xyz: ordering already correct (['Nb', 'N'])
ScN_108_slab.xyz: ordering already correct (['Sc', 'N'])
TiN_108_slab.xyz: ordering already correct (['Ti', 'N'])
VN_108_slab.xyz: ordering already correct (['V', 'N'])
ZrN_108_slab.xyz: ordering already correct (['Zr', 'N'])
NbN_144_slab.xyz: ordering already correct (['Nb', 'N'])
ScN_144_slab.xyz: ordering already correct (['Sc', 'N'])
TiN_144_slab.xyz: ordering already correct (['Ti', 'N'])
VN_144_slab.xyz: ordering already correct (['V', 'N'])
ZrN_144_slab.xyz: ordering already correct (['Zr', 'N'])
NbN_192_slab.xyz: rewritten with order ['Nb', 'N']
ScN_192_slab.xyz: rewritten with order ['Sc', 'N']
TiN_192_slab.xyz: rewritten with order ['Ti', 'N']
VN_192_slab.xyz: rewritten with order ['V', 'N']
ZrN_192_slab.xyz: rewritten with order ['Zr', 'N']
NbN_256_slab.xyz: rewritten with order ['Nb', 'N']
ScN_256_slab.xyz: rewritten with order ['Sc', 'N']
TiN_256_slab.xyz: rewritten with order ['Ti', 'N']
VN_256_slab.xyz: rewritten w

In [10]:
# FULL SLAB MAKER 3x3 3 Layer
from ase.io import read,write
from ase.visualize import view
from ase.build import surface
cif_dir = '../cif/output_file'
metals = ['TiN','VN','ZrN','ScN','NbN']

for metal in metals:
    cif = read(f'{cif_dir}/{metal}_relaxed.extxyz')

    slab = surface(cif, (1, 0, 0), layers=2, vacuum=10.0)
    slab = slab.repeat((3, 3, 1))

    z_coords = slab.get_positions()[:, 2]
    top_indices = [i for i, z in enumerate(z_coords) if z > 16.0]
    if top_indices:
        # delete the stray top layer atoms starting from the highest index
        for idx in sorted(top_indices, reverse=True):
            del slab[idx]
        print(f'Pruned {metal}: removed {len(top_indices)} atoms above z=16')

    out_dir = Path('./input_slab_108')
    out_dir.mkdir(exist_ok=True)
    write(out_dir / f'{metal}_108_slab.xyz', slab)

Pruned TiN: removed 36 atoms above z=16
Pruned VN: removed 36 atoms above z=16
Pruned ZrN: removed 36 atoms above z=16
Pruned ScN: removed 36 atoms above z=16
Pruned NbN: removed 36 atoms above z=16


In [13]:
# FULL SLAB MAKER 4x4 3 Layer
from ase.io import read,write
from ase.visualize import view
from ase.build import surface
cif_dir = '../cif/output_file'
metals = ['TiN','VN','ZrN','ScN','NbN']

for metal in metals:
    cif = read(f'{cif_dir}/{metal}_relaxed.extxyz')

    slab = surface(cif, (1, 0, 0), layers=2, vacuum=10.0)
    slab = slab.repeat((4, 4, 1))

    z_coords = slab.get_positions()[:, 2]
    top_indices = [i for i, z in enumerate(z_coords) if z > 16.0]
    if top_indices:
        # delete the stray top layer atoms starting from the highest index
        for idx in sorted(top_indices, reverse=True):
            del slab[idx]
        print(f'Pruned {metal}: removed {len(top_indices)} atoms above z=16')

    out_dir = Path('./input_slab_192')
    out_dir.mkdir(exist_ok=True)
    write(out_dir / f'{metal}_192_slab.xyz', slab)

Pruned TiN: removed 64 atoms above z=16
Pruned VN: removed 64 atoms above z=16
Pruned ZrN: removed 64 atoms above z=16
Pruned ScN: removed 64 atoms above z=16
Pruned NbN: removed 64 atoms above z=16


In [14]:
# FULL SLAB MAKER 4x4 4 Layer
from ase.io import read,write
from ase.visualize import view
from ase.build import surface
cif_dir = '../cif/output_file'
metals = ['TiN','VN','ZrN','ScN','NbN']

for metal in metals:
    cif = read(f'{cif_dir}/{metal}_relaxed.extxyz')

    slab = surface(cif, (1, 0, 0), layers=2, vacuum=10.0)
    slab = slab.repeat((4, 4, 1))

    out_dir = Path('./input_slab_256')
    out_dir.mkdir(exist_ok=True)
    write(out_dir / f'{metal}_256_slab.xyz', slab)

In [ ]:
## PYMATGEN




### Slab analysis (pymatgen)
Run a quick analysis on `slab/input_slab108/TiN_108_slab.xyz` using pymatgen's `SlabGenerator` — reports polarity, symmetry, thickness, vacuum and top/bottom terminations.

In [2]:
from ase.io import read
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.surface import SlabGenerator


def analyze_slab_from_xyz(xyz_path, miller=(1, 0, 0), min_slab=15, min_vac=5, center_slab=True, symmetrize=True, top_tol=0.5):
    """Read an ASE .xyz (extxyz with lattice), convert to pymatgen.Structure and run SlabGenerator.

    Returns a list of dicts with simple diagnostics + the pymatgen Slab object under key 'slab'.
    """
    atoms = read(xyz_path)
    structure = AseAtomsAdaptor.get_structure(atoms)

    sg = SlabGenerator(structure, miller, min_slab, min_vac, center_slab=center_slab)
    slabs = sg.get_slabs(symmetrize=symmetrize)

    out = []
    for slab in slabs:
        z = [site.coords[2] for site in slab.sites]
        zmin, zmax = min(z), max(z)
        thickness = zmax - zmin
        vacuum = slab.lattice.c - thickness

        bottom = sorted({site.specie.symbol for site in slab.sites if site.coords[2] - zmin < top_tol})
        top = sorted({site.specie.symbol for site in slab.sites if zmax - site.coords[2] < top_tol})

        out.append({
            "miller": slab.miller_index,
            "is_polar": slab.is_polar(),
            "is_symmetric": slab.is_symmetric(),
            "thickness_A": round(thickness, 6),
            "vacuum_A": round(vacuum, 6),
            "bottom_termination": bottom,
            "top_termination": top,
            "slab": slab,
        })
    return out


# --- run it on the file you specified ---
path = "./input_slab108/TiN_108_slab.xyz"
results = analyze_slab_from_xyz(path)

if not results:
    print("No slabs generated")
else:
    for i, info in enumerate(results, 1):
        print(f"slab {i}: miller={info['miller']}  polar={info['is_polar']}  symmetric={info['is_symmetric']}")
        print(f"        thickness={info['thickness_A']:.3f} Å   vacuum={info['vacuum_A']:.3f} Å")
        print(f"        bottom={info['bottom_termination']}  top={info['top_termination']}")

# keep `results` for later use / visualization in the notebook
results

slab 1: miller=(1, 0, 0)  polar=False  symmetric=True
        thickness=23.635 Å   vacuum=15.041 Å
        bottom=['N', 'Ti']  top=['N', 'Ti']


[{'miller': (1, 0, 0),
  'is_polar': False,
  'is_symmetric': True,
  'thickness_A': np.float64(23.635407),
  'vacuum_A': np.float64(15.040714),
  'bottom_termination': ['N', 'Ti'],
  'top_termination': ['N', 'Ti'],
  'slab': Structure Summary
  Lattice
      abc : 4.297346799 26.446020193899997 38.67612119099999
   angles : 90.0 90.0 90.0
   volume : 4395.453000562735
        A : np.float64(4.297346799) np.float64(0.0) np.float64(2.631366001110737e-16)
        B : np.float64(4.25284440676163e-15) np.float64(26.446020193899997) np.float64(1.619351699032295e-15)
        C : np.float64(0.0) np.float64(0.0) np.float64(38.67612119099999)
      pbc : True True True
  PeriodicSite: N (2.149, 10.0, 18.26) [0.5, 0.3781, 0.4722]
  PeriodicSite: N (1e-09, 10.0, 7.52) [2.327e-10, 0.3781, 0.1944]
  PeriodicSite: N (1e-09, 12.15, 18.26) [2.327e-10, 0.4594, 0.4722]
  PeriodicSite: N (2.149, 12.15, 7.52) [0.5, 0.4594, 0.1944]
  PeriodicSite: N (2.149, 14.3, 18.26) [0.5, 0.5406, 0.4722]
  PeriodicSite

In [ ]:
# Quick check: adapt ASE slab -> pymatgen and report polarity/symmetry
from ase.io import read
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.surface import SlabGenerator

p = "slab/input_slab108/TiN_108_slab.xyz"
atoms = read(p)
structure = AseAtomsAdaptor.get_structure(atoms)

# Generate slab objects from the provided (already-slab) structure and pick the best match
slabs = SlabGenerator(structure, (1, 0, 0), min_slab=1.0, min_vac=1.0, center_slab=False).get_slabs(symmetrize=False)
if not slabs:
    raise RuntimeError(f"SlabGenerator returned no slabs for {p}")

# choose slab with thickness closest to the input
def thickness(s):
    zs = [site.coords[2] for site in s.sites]
    return max(zs) - min(zs)

target_thick = thickness(structure) if hasattr(structure, 'sites') else None
slab = min(slabs, key=lambda s: abs(thickness(s) - (target_thick or thickness(slabs[0]))))

print(f"File: {p}")
print("is_polar:", slab.is_polar())
print("is_symmetric:", slab.is_symmetric())
print(f"thickness = {thickness(slab):.3f} Å   vacuum (cell c - thickness) = {slab.lattice.c - thickness(slab):.3f} Å")

# keep `slab` available for further inspection
slab